In [1]:
############################################################
# STEP3.2_03b_fix_nan_scores.R
#
# 目的: 
#   k_score_*.csv の中に NA/NaN が含まれているファイルを検出し、
#   その条件のみ NbClust を再実行して修復を試みる。
#   修復できない場合は、原因（どのkでエラーが出ているか）を表示する。
############################################################

suppressPackageStartupMessages({
  library(NbClust)
  library(dplyr)
  library(stringr)
})

set.seed(42)

# ---- 設定 (これまでのステップと同じにする) ----
root_base <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127"
sub_base  <- file.path(root_base, "sub")

# 対象データセット
datasets_variables <- c("OH", "FP")
datasets_samples   <- c("A_OH_plus_others", "B_FP_plus_others", "C_OH_only", "D_FP_only")

# MDS モード
mode_dims <- c("linear_top3", "linear_cumeig", "nonlinear_top3", "nonlinear_cumeig")

# NbClust 指標
index_list <- c("silhouette","dunn","gap","ch","db","ptbiserial")

# NbClust 設定
min_nc <- 2
max_nc <- 25

# ★ run_id を指定 (エラーが出ているID)
override_run_id <- "20251130_153500" 

# ---- パス設定 ----
run_id <- override_run_id
step02_base <- file.path(sub_base, "02_mds_STEP3.2_signlessCorr")
step03_base <- file.path(sub_base, "03_clustering_STEP3.2_signlessCorr")

step02_run_dir <- file.path(step02_base, paste0("run_", run_id))
step03_run_dir <- file.path(step03_base, paste0("run_", run_id))

cat(">>> STEP3.2_03b_fix_nan_scores — run_id:", run_id, "\n")

# ---- ヘルパー関数: MDS座標読み込み ----
read_coords_safe <- function(unit, dataset, cond) {
  # ファイル名パターン: MDScoords_<cond>_<unit>_<dataset>_<runid>.csv
  fname <- sprintf("MDScoords_%s_%s_%s_%s.csv", cond, unit, dataset, run_id)
  path <- file.path(step02_run_dir, unit, dataset, fname)
  
  if (!file.exists(path)) {
    # 旧パターンもチェック
    fname_old <- sprintf("MDS_%s_%s_%s.csv", cond, dataset, run_id)
    path_old <- file.path(step02_run_dir, unit, dataset, fname_old)
    if(file.exists(path_old)) return(as.matrix(read.csv(path_old, row.names=1)))
    return(NULL)
  }
  return(as.matrix(read.csv(path, row.names=1)))
}

# ---- メイン処理: 診断と修復 ----
check_and_fix <- function(unit, dataset) {
  
  indices_dir <- file.path(step03_run_dir, unit, dataset, "indices")
  if (!dir.exists(indices_dir)) return()
  
  for (cond in mode_dims) {
    for (idx in index_list) {
      
      # 1. k_score ファイルの確認
      csv_name <- sprintf("k_score_%s_%s_%s_%s_%s.csv", cond, unit, dataset, idx, run_id)
      csv_path <- file.path(indices_dir, csv_name)
      
      if (!file.exists(csv_path)) next
      
      df_score <- read.csv(csv_path)
      
      # NaN または NA が含まれているかチェック
      if (any(is.na(df_score$score)) || any(is.nan(df_score$score))) {
        
        cat("\n[DETECTED] NaN found in:", csv_name, "\n")
        
        # どの k がダメなのか表示
        bad_rows <- df_score[is.na(df_score$score), ]
        cat("   -> Missing scores at k =", paste(bad_rows$k, collapse=","), "\n")
        
        # 2. 再計算を試みる
        cat("   -> Attempting Re-calculation...\n")
        
        coords <- read_coords_safe(unit, dataset, cond)
        if (is.null(coords)) {
          cat("   -> [ERROR] MDS Coords not found. Cannot recalculate.\n")
          next
        }
        
        # 2次元以上あるか
        if (ncol(coords) < 2) {
          cat("   -> [ERROR] Dimensions < 2. Cannot calculate NbClust.\n")
          next
        }
        coords_2d <- coords[, 1:2] # NbClustは基本2次元を使うことが多いがデータ次第
        
        # NbClust 再実行
        res <- NULL
        tryCatch({
          res <- NbClust(
            data = coords_2d,
            distance = "euclidean",
            min.nc = min_nc,
            max.nc = min(max_nc, nrow(coords_2d) - 1),
            method = "ward.D2",
            index = idx
          )
        }, error = function(e) {
          cat("   -> [NBCLUST ERROR]", e$message, "\n")
        })
        
        # 3. 結果の検証と保存
        if (!is.null(res) && !is.null(res$All.index)) {
          new_scores <- as.numeric(res$All.index)
          
          # 再度 NaN チェック
          if (any(is.na(new_scores))) {
            cat("   -> [FAILURE] Re-calculation also produced NaN. (Mathematical limitation)\n")
            cat("      Cause: Index '", idx, "' may be undefined for this data distribution.\n", sep="")
            
            # 応急処置: NaN を除去して保存（Pythonでのエラーを防ぐため）
            valid_k <- seq_along(new_scores)[!is.na(new_scores)]
            valid_s <- new_scores[!is.na(new_scores)]
            
            if (length(valid_s) > 0) {
              df_fixed <- data.frame(k = valid_k, score = valid_s)
              write.csv(df_fixed, csv_path, row.names = FALSE)
              cat("   -> [FIXED] Overwritten file with VALID scores only (NaN rows removed).\n")
            } else {
              cat("   -> [CRITICAL] All scores are NaN. Deleting file to avoid errors.\n")
              unlink(csv_path)
            }
            
          } else {
            # 正常に計算できた場合
            k_seq <- seq_along(new_scores)
            # NbClustの仕様で kの始まりが min.nc と一致しない場合があるので補正
            # (通常 All.index は min.nc からスタートするはずだが、ここでは単純連番か確認)
            # 実際には names(res$All.index) を見るのが確実
            
            if (!is.null(names(res$All.index))) {
               k_vals <- as.numeric(names(res$All.index))
            } else {
               k_vals <- min_nc:(min_nc + length(new_scores) - 1)
            }
            
            df_new <- data.frame(k = k_vals, score = new_scores)
            write.csv(df_new, csv_path, row.names = FALSE)
            cat("   -> [SUCCESS] Re-calculation successful. File overwritten.\n")
          }
        } else {
          cat("   -> [FAILURE] NbClust returned NULL.\n")
        }
      }
    }
  }
}

# ---- 実行ループ ----

cat("\n=== Checking Variables (OH/FP) ===\n")
for (ds in datasets_variables) {
  check_and_fix("variables", ds)
}

cat("\n=== Checking Samples (A-D) ===\n")
for (ds in datasets_samples) {
  check_and_fix("samples", ds)
}

cat("\n✅ Check & Fix finished.\n")

>>> STEP3.2_03b_fix_nan_scores <U+2014> run_id: 20251130_153500 

=== Checking Variables (OH/FP) ===

[DETECTED] NaN found in: k_score_linear_top3_variables_OH_silhouette_20251130_153500.csv 
   -> Missing scores at k = 2 
   -> Attempting Re-calculation...
   -> [SUCCESS] Re-calculation successful. File overwritten.

[DETECTED] NaN found in: k_score_linear_top3_variables_OH_dunn_20251130_153500.csv 
   -> Missing scores at k = 3 
   -> Attempting Re-calculation...
   -> [SUCCESS] Re-calculation successful. File overwritten.

[DETECTED] NaN found in: k_score_linear_top3_variables_OH_gap_20251130_153500.csv 
   -> Missing scores at k = 2 
   -> Attempting Re-calculation...
   -> [SUCCESS] Re-calculation successful. File overwritten.

[DETECTED] NaN found in: k_score_linear_top3_variables_OH_ch_20251130_153500.csv 
   -> Missing scores at k = 25 
   -> Attempting Re-calculation...
   -> [SUCCESS] Re-calculation successful. File overwritten.

[DETECTED] NaN found in: k_score_linear_top3_v


=== Checking Samples (A-D) ===

[DETECTED] NaN found in: k_score_linear_top3_samples_A_OH_plus_others_silhouette_20251130_153500.csv 
   -> Missing scores at k = 2 
   -> Attempting Re-calculation...
   -> [SUCCESS] Re-calculation successful. File overwritten.

[DETECTED] NaN found in: k_score_linear_top3_samples_A_OH_plus_others_dunn_20251130_153500.csv 
   -> Missing scores at k = 22 
   -> Attempting Re-calculation...
   -> [SUCCESS] Re-calculation successful. File overwritten.

[DETECTED] NaN found in: k_score_linear_top3_samples_A_OH_plus_others_gap_20251130_153500.csv 
   -> Missing scores at k = 2 
   -> Attempting Re-calculation...
   -> [SUCCESS] Re-calculation successful. File overwritten.

[DETECTED] NaN found in: k_score_linear_top3_samples_A_OH_plus_others_ch_20251130_153500.csv 
   -> Missing scores at k = 25 
   -> Attempting Re-calculation...
   -> [SUCCESS] Re-calculation successful. File overwritten.

[DETECTED] NaN found in: k_score_linear_top3_samples_A_OH_plus_oth


<U+2705> Check & Fix finished.
